<div style="background:linear-gradient(135deg,#1b1448 0%,#3b1f5e 55%,#7c2d12 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#fed7aa;font-weight:700;text-transform:uppercase">Chapter 1 · What building AI with AI means</div>
  <div style="font-size:34px;font-weight:900;line-height:1.1;margin:10px 0 6px">What building AI with AI means</div>
  <div style="font-size:15px;color:#ffe4d6;max-width:740px;line-height:1.6">A language model as a component of a conventional program, and an assistant whose code is verified by tests, both run on eight support tickets.</div>
  <div style="font-size:13px;color:#fed7aa;margin-top:14px">Building AI with AI: A Visual Handbook · John Fisher</div>
</div>

This notebook runs the chapter's two ideas on eight support tickets for an online store. First a language model classifies each ticket, as a component behind a single function call. Then the model writes a small helper function, and test cases written beforehand decide whether it is accepted. A final section totals what the calls cost.

## Setup

This cell installs Anthropic's Python library and builds the client every call below goes through. It replays recorded answers unless you set `LIVE = True`.

In [1]:
# Set LIVE = True to send these requests to the Anthropic API with your own key, saved in
# Colab's secrets panel as ANTHROPIC_API_KEY. Left False, the notebook replays the answers
# recorded for the book, with no key and no cost.
LIVE = False
%pip install -q --disable-pip-version-check anthropic==2.7.0
import importlib, pathlib, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/chapters/notebooks/"
    pathlib.Path("recordings").mkdir(exist_ok=True)
    for name in ["book_client.py", "recordings/what-building-ai-with-ai-means.json"]:
        urllib.request.urlretrieve(BASE + name, name)
import book_client
importlib.reload(book_client)
client = book_client.make_client("what-building-ai-with-ai-means", mode="live" if LIVE else None)

Note: you may need to restart the kernel to use updated packages.


Replaying 9 recorded responses from claude-sonnet-5-5, recorded 2026-11-10. Set LIVE = True in Setup to call the API yourself.


## A model as a component

`triage` looks like any other function. It takes a ticket and returns a category. Inside, it sends the ticket to the model with a short instruction, the *system prompt*, that lists the four allowed categories, and reads the word that comes back. The rest of the program never needs to know a model was involved, and that is what makes the model a component.

A reply's `content` is a list of blocks, and `answer_text` keeps only the text blocks. The last line of `triage` is a guard. A reply that isn't one of the four categories comes back as `unrecognized`, so it can't pass for a category further on.

In [2]:
# Eight support tickets for an online store
TICKETS = [
    "I was charged twice for my March order and only got one receipt.",
    "The reset link in your email says it has expired, and I still can't sign in.",
    "Tracking has said 'label created' for six days. Where is my package?",
    "Do you sell gift cards that can be printed at home?",
    "My card was declined at checkout, but my bank says the payment went through.",
    "I changed my email address and now the app won't accept my password.",
    "The box arrived crushed and two of the mugs inside are broken.",
    "I can't log in to download the invoice for my last order.",
]
CATEGORIES = ["billing", "login", "shipping", "other"]
MODEL = "claude-sonnet-5-5"
SYSTEM = ("You sort customer support tickets for an online store. "
          "Reply with exactly one word from this list: " + ", ".join(CATEGORIES) + ".")
USAGE = []   # the token counts of every call, added up in the cost section


def answer_text(reply):
    """The text of a reply. Its content is a list of blocks, and the text blocks hold the answer."""
    return "".join(block.text for block in reply.content if block.type == "text")

In [3]:
def triage(ticket):
    """Ask the model for the ticket's category. A reply outside the list comes back as 'unrecognized'."""
    reply = client.messages.create(model=MODEL, max_tokens=1024, system=SYSTEM,
                                   messages=[{"role": "user", "content": ticket}])
    USAGE.append(reply.usage)
    label = answer_text(reply).strip().lower()
    return label if label in CATEGORIES else "unrecognized"


for ticket in TICKETS:
    print(f"{triage(ticket):<10} {ticket}")

billing    I was charged twice for my March order and only got one receipt.
login      The reset link in your email says it has expired, and I still can't sign in.
shipping   Tracking has said 'label created' for six days. Where is my package?
other      Do you sell gift cards that can be printed at home?
billing    My card was declined at checkout, but my bank says the payment went through.
login      I changed my email address and now the app won't accept my password.
shipping   The box arrived crushed and two of the mugs inside are broken.
login      I can't log in to download the invoice for my last order.


Every ticket landed in an allowed category, so the guard never fired. Six of the eight have one plain answer. The other two are judgment calls. The crushed box went to `shipping`, though it could also be read as a complaint about damaged goods, and the ticket about downloading an invoice went to `login`, though what the customer wants is a billing document. Neither label is unreasonable, and a program that routes tickets on these labels inherits both judgments. Part 3 measures a classifier like this one against a test set before anyone relies on it.

## An assistant that writes code you check with tests

Now the model writes code instead of sorting tickets. The request asks for `normalize_label`, a function that turns a messy one-word reply such as `" Billing.\n"` into a clean category. The six cases that check it were written by the book before the model's code was read. They include a capitalized word, trailing punctuation, a word outside the list and an empty string.

In [4]:
REQUEST = """Write a Python function normalize_label(raw: str) -> str.
It takes a model's one-word reply, such as " Billing.\n", and returns a clean category.
Lowercase it, strip whitespace and any trailing punctuation, and return the result if it is one of
billing, login, shipping or other. Return "other" for anything else.
Reply with only the code, in one Python code block, with no explanation."""

reply = client.messages.create(model=MODEL, max_tokens=2048,
                               messages=[{"role": "user", "content": REQUEST}])
USAGE.append(reply.usage)
written = answer_text(reply)
print(written)

```python
import string

VALID_LABELS = {"billing", "login", "shipping", "other"}


def normalize_label(raw: str) -> str:
    if not isinstance(raw, str):
        return "other"

    cleaned = raw.strip().lower()
    cleaned = cleaned.strip(string.punctuation + string.whitespace)

    return cleaned if cleaned in VALID_LABELS else "other"
```


The reply is one fenced code block, as the request asked. The function lowercases the reply, strips whitespace and punctuation from both ends, and returns the result when it is one of the four labels, or `other` when it isn't. It also returns `other` for anything that isn't a string, which the request didn't ask for. Read code like this before you run it, because the next cell runs it with `exec`, which would run anything the model had written.

In [5]:
import re

# Take the code out of its fenced block, then run it. Run code a model wrote only after reading it.
match = re.search(r"```(?:python)?\n(.*?)```", written, re.S)
namespace = {}
exec(match.group(1) if match else written, namespace)
normalize_label = namespace["normalize_label"]

# These cases are the book's, written before the model's code was read. They are the contract.
CASES = [(" Billing.\n", "billing"), ("LOGIN", "login"), ("shipping!", "shipping"),
         ("other", "other"), ("refund", "other"), ("", "other")]
passed = 0
for raw, expected in CASES:
    got = normalize_label(raw)
    passed += got == expected
    print(f"{'pass' if got == expected else 'FAIL'}  normalize_label({raw!r}) -> {got!r}, expected {expected!r}")
print(f"{passed} of {len(CASES)} cases pass")

pass  normalize_label(' Billing.\n') -> 'billing', expected 'billing'
pass  normalize_label('LOGIN') -> 'login', expected 'login'
pass  normalize_label('shipping!') -> 'shipping', expected 'shipping'
pass  normalize_label('other') -> 'other', expected 'other'
pass  normalize_label('refund') -> 'other', expected 'other'
pass  normalize_label('') -> 'other', expected 'other'
6 of 6 cases pass


All six cases pass, so the function is accepted into the program. Had one failed, the fix would start from the failing case, by asking again with that case in the request or by correcting the code by hand, and the cases would decide again. This is the second sense of the book's title. An assistant writes the code, and you accept what passes the tests you wrote yourself. Part 2 develops this workflow with Claude Code.

## What the calls cost

Every reply reports how many tokens it read and wrote, in `reply.usage`, and the API bills by those counts. The notebook kept each call's usage in `USAGE`. The prices in the cell are Claude Sonnet 5.5's as of November 2026, 2 dollars per million input tokens and 10 dollars per million output tokens, so check the current ones before relying on the total.

In [6]:
# Claude Sonnet 5.5's prices per million tokens, as of November 2026. Check the current ones before relying on them.
PRICE_IN, PRICE_OUT = 2.00, 10.00
tokens_in = sum(u.input_tokens for u in USAGE)
tokens_out = sum(u.output_tokens for u in USAGE)
print(f"{len(USAGE)} calls, {tokens_in} input tokens, {tokens_out} output tokens")
print(f"about {tokens_in / 1e6 * PRICE_IN + tokens_out / 1e6 * PRICE_OUT:.4f} dollars at those prices\n")

# Output tokens per ticket, and how many of them the model spent thinking before it answered
for ticket, usage in zip(TICKETS, USAGE):
    thinking = usage.output_tokens_details.thinking_tokens
    print(f"{usage.output_tokens:>3} out, {thinking:>2} thinking   {ticket}")

9 calls, 665 input tokens, 337 output tokens
about 0.0047 dollars at those prices

  4 out,  0 thinking   I was charged twice for my March order and only got one receipt.
  3 out,  0 thinking   The reset link in your email says it has expired, and I still can't sign in.
  5 out,  0 thinking   Tracking has said 'label created' for six days. Where is my package?
 93 out, 90 thinking   Do you sell gift cards that can be printed at home?
  4 out,  0 thinking   My card was declined at checkout, but my bank says the payment went through.
  3 out,  0 thinking   I changed my email address and now the app won't accept my password.
 48 out, 43 thinking   The box arrived crushed and two of the mugs inside are broken.
 34 out, 31 thinking   I can't log in to download the invoice for my last order.


Nine calls read 665 tokens and wrote 337, about half a cent at those prices. The breakdown holds a surprise. Five tickets were answered in 3 to 5 output tokens, but the gift cards, the crushed box and the invoice took 93, 48 and 34. The difference is thinking. On those three the model reasoned before it answered, 90, 43 and 31 tokens of it, and that reasoning is billed as output although its text isn't shown. The reply carries it as a `thinking` block with an empty `thinking` field, which is why `answer_text` reads only the text blocks. The tickets that needed a judgment call cost the most, and Chapter 3 returns to tokens and cost.

## Where building AI with AI goes wrong

A model's reply is text, and nothing forces it to be one of your categories. The cell below takes four made-up replies, of the kinds a model can return, through the strict check in `triage` and through the model's own `normalize_label`.

In [7]:
# The same raw replies through triage's strict check and through the model's normalize_label
for raw in ["billing", "Billing.", "refund", "I think this is a login problem."]:
    strict = raw.strip().lower()
    strict = strict if strict in CATEGORIES else "unrecognized"
    print(f"{raw!r:36} strict: {strict:<13} normalize_label: {normalize_label(raw)}")

'billing'                            strict: billing       normalize_label: billing
'Billing.'                           strict: unrecognized  normalize_label: billing
'refund'                             strict: unrecognized  normalize_label: other
'I think this is a login problem.'   strict: unrecognized  normalize_label: other


The strict check accepts only an exact category, so `Billing.` with a capital and a period comes back `unrecognized`, and a person or a retry has to deal with it. `normalize_label` turns the same reply into `billing`. It also turns `refund` and a whole sentence into `other`, which looks tidy and hides that the model didn't answer the question it was asked. The strict check stops work that a little cleanup would have saved. The normalizer files real problems under `other` without saying so. A system built on a model has to choose which failure it can live with, and count how often each one happens. None of this showed in the eight tickets above, where every reply was clean, so one good run says little about the next thousand.

## The triage module from the chapter's Claude Code build

The chapter's build gave Claude Code a spec for `triage.py` and seventeen tests, and it generated the module these cells load. Its `triage` takes the client as an argument, and the spec fixes its request to match the one the `triage` above sends, with the same model, `max_tokens`, system prompt and message.

In [8]:
# triage.py is the module Claude Code generated in this chapter's build, from the spec and tests in
# builds/what-building-ai-with-ai-means/. Colab fetches it from the book's repository.
BUILD = "what-building-ai-with-ai-means"
if "google.colab" in sys.modules:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/builds/"
                               f"{BUILD}/generated/triage.py", "triage.py")
else:
    sys.path.insert(0, str(pathlib.Path("../../builds", BUILD, "generated").resolve()))
import triage as built
built = importlib.reload(built)

In [9]:
# The generated function, on the same eight tickets
for ticket in TICKETS:
    print(f"{built.triage(client, ticket):<10} {ticket}")

billing    I was charged twice for my March order and only got one receipt.
login      The reset link in your email says it has expired, and I still can't sign in.
shipping   Tracking has said 'label created' for six days. Where is my package?
other      Do you sell gift cards that can be printed at home?
billing    My card was declined at checkout, but my bank says the payment went through.
login      I changed my email address and now the app won't accept my password.
shipping   The box arrived crushed and two of the mugs inside are broken.
login      I can't log in to download the invoice for my last order.


The generated `triage` returned the same category as the function above for all eight tickets. Its requests are identical down to the system prompt and `max_tokens`, so the notebook answers them from the same recording, and a request that differed in any field would have found no recorded response and raised an error. The recording works as a second check on the generated code, alongside the seventeen tests it was built against.

## Recap

- A model can be a component of a conventional program. `triage` hides the call behind a function, and a guard keeps a reply outside the list from passing as a category.
- Two of the eight labels were judgment calls, and only a test set can say how often such calls go the right way.
- A model can also write code. The book's six cases checked `normalize_label` before it was used, and all six passed.
- Replies report their token counts. The nine calls cost about half a cent at November 2026 prices, and the three hardest tickets cost the most because the model thought before answering.
- Claude Code generated `triage.py` from a spec and seventeen tests, and it sends the same requests and returns the same categories as the function written by hand.
- The next chapter, How a language model produces text, opens up the model behind these replies.

---
<div style="text-align:center;color:#666f8e;font-size:12px;margin-top:10px">Building AI with AI: A Visual Handbook · © 2026 John Fisher · text CC BY 4.0, code MIT</div>